In [ ]:
import itertools
from pathlib import Path
from rich import print as rprint
import polars as pl
from tqdm import tqdm

In [ ]:
list_df = []
for path_file in Path("/u501/y4466wan/projects/2025/verb-overgen/data/zorro/zorro/sentences/babyberta").iterdir():
    with open(path_file, "r") as f:
        ser_setence = pl.Series("sentence", f)
        df = pl.DataFrame({"sentence": ser_setence})
        df = df.with_row_index("file_line_idx", 1).with_columns(
            file_name = pl.lit(path_file.stem),
            is_grammatical = (pl.col("file_line_idx") % 2 == 0),
        ).select(
            "file_name",
            "file_line_idx",
            "is_grammatical",
            "sentence",
        )

        list_df.append(df)

In [ ]:
dataset = pl.concat(list_df)

In [ ]:
len(dataset)

In [ ]:
from verb_overgen.prep.lemmatize_data import text_to_lemma, lemmatize_dtype

In [ ]:
def update_pbar(pbar, func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        pbar.update(1)
        return result
    return wrapper

In [ ]:
pbar_lemmatize = tqdm(desc="Lemmatizing", total=len(dataset))
dataset = dataset.with_columns(
    pl.col("sentence").map_elements(update_pbar(pbar_lemmatize, text_to_lemma), lemmatize_dtype).struct.unnest()
)

dataset.write_parquet("/u501/y4466wan/projects/2025/verb-overgen/data/zorro/zorro_babyberta_lemmatized.parquet")